In [0]:
%scala
import java.time.LocalDate
import java.time.LocalDateTime
import org.apache.spark.sql.{SparkSession, Row}
import org.apache.spark.sql.types._
import scala.util.Try
import scala.collection.JavaConverters._
import org.apache.spark.sql.functions.col
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.DateType
import java.time.format.DateTimeFormatter
import org.apache.spark.sql.{DataFrame, Row}
import org.apache.spark.sql.types.{StructType, StructField, StringType}
import org.apache.spark.sql.Column
import org.apache.spark.sql.functions._
import java.time._
import java.time.{Instant, ZoneId, ZonedDateTime}







In [0]:
%scala


//parametros de entrada
var parsing_in = dbutils.widgets.get("parsing_in")
var nombreArchivo = dbutils.widgets.get("nombreArchivo")
var parsing_out = dbutils.widgets.get("parsing_out")




In [0]:
/*
var parsing_in = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/campanas/marketing/rut_emergencia/stage/tmp"
//var pathStage = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/campanas/marketing/rut_emergencia/stage/"
var nombreArchivo = "CARGA_RUT_pdmollma_20250421.txt"
var parsing_out = "abfss://ingestas@stbigdatadev02.dfs.core.windows.net/data/interacciones/campanas/marketing/rut_emergencia/stage/"
*/

In [0]:
// Revisar las rutas para rescatar la data y dejarla en dataframes temporales, en el caso que la ruta no exista o no tenga data se crea un dataframe vacío con el esquema de la data.

val spark = SparkSession.builder().appName("agregaColumna").getOrCreate()


// Función para verificar si la ruta tiene archivos
def checkPath(path: String): Boolean = Try(dbutils.fs.ls(path)).getOrElse(Seq()).nonEmpty


// Verificar si las rutas tienen datos y cargar los DataFrames
val dfr1 = if (checkPath(parsing_in)) {
  println(s"La ruta $parsing_in tiene archivos.")
  spark.read.format("csv")
    .option("delimiter", "\\t")
    .option("header", "false")
    .option("inferSchema", "true")
    .load(parsing_in)
} else {
  throw new RuntimeException(s"La ruta $parsing_in no existe o está vacía. Creando DataFrame vacío...")
  //spark.createDataFrame(emptyList, schema)
}

// Eliminamos duplicados
val df2 = dfr1.dropDuplicates()

var largo = df2.columns.length
// Agrega Columna con nombre archivo
val dfWithNewColumn = df2.withColumn(s"_c${largo}", lit(nombreArchivo))

//dfWithNewColumn.printSchema()
//display(dfWithNewColumn)

In [0]:
%scala


// Función para limpiar columnas String
def cleanStringColumn(colName: String): Column = {
  val trimmedColumn = trim(col(colName))
  when(trimmedColumn === "" || trimmedColumn === "null" || col(colName).isNull, null)
    .otherwise(trimmedColumn)
    .alias(colName)
}

try {

  
    val df_final = dfWithNewColumn.dropDuplicates()

val filename = nombreArchivo

  // Guarda el archivo en la carpeta de salida
  df_final.coalesce(1).write.option("header", "false").option("delimiter", "\\t").mode("overwrite").csv(parsing_out)

// Renombrar el archivo guardado con el filename construido
val files = dbutils.fs.ls(parsing_out)
val oldFilePathOption = files.find(file => file.name.startsWith("part-"))

oldFilePathOption match {
  case Some(oldFile) =>
    val oldFilePath = oldFile.path
    val newFilePath = parsing_out + filename
    dbutils.fs.mv(oldFilePath, newFilePath)
    println(s"El archivo ha sido renombrado a: $filename")
  case None =>
    println("No se encontró ningún archivo que cumpla con el criterio.")
}
} catch {
  case e: Exception =>
    println("[ERROR] " + e)
    throw e
}

println("[INFO] PROCESO TERMINADO")

In [0]:
%scala
val files = dbutils.fs.ls(parsing_in)

// Función para eliminar archivos y carpetas temporales que se utilizaron en el proceso.
def deleteFilesAndFolders(path: String): Unit = {
  val filesAndDirs = dbutils.fs.ls(path)

  filesAndDirs.foreach(file => {
    if (file.isFile) {
      dbutils.fs.rm(file.path, true)
    } 
    else if (file.isDir) {
      deleteFilesAndFolders(file.path)
    }
  })
  
  dbutils.fs.rm(path, true)
}

deleteFilesAndFolders(parsing_in)
